# Notebook 02: Dataset build and inspection

This notebook reproduces the Phase 3 dataset build from the real `scripts/build_phase3_dataset.py` entrypoint, inspects the generated `manifest.json`, and shows how raw downloaded corpora stay outside analyzer logic until the manifest pipeline materializes a deterministic dataset.

In [8]:
from pathlib import Path
import json
import shutil
import subprocess
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'rtl_analyzer').exists():
            return candidate
    raise RuntimeError('Could not locate repository root from notebook working directory')


repo_root = find_repo_root(Path.cwd().resolve())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


def repo_relative(path: Path) -> str:
    try:
        return path.relative_to(repo_root).as_posix()
    except ValueError:
        return path.as_posix()


def run_command(args: list[str]) -> subprocess.CompletedProcess[str]:
    result = subprocess.run(
        args,
        cwd=repo_root,
        check=False,
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr or result.stdout or f'command failed: {args}')
    return result


print(repo_relative(repo_root))

.


## Build the bootstrap dataset with the production script

The notebook uses fixture-backed synthetic and external corpora so the workflow stays runnable everywhere. The important contract is the same as the real pipeline: corpora live in source directories, `scripts/build_phase3_dataset.py` assigns grouped splits, copies only the selected files into an output dataset root, and writes a portable `manifest.json`.

In [9]:
from rtl_analyzer.ml.dataset_manifest import read_manifest

fixtures_root = repo_root / 'tests' / 'fixtures'
synthetic_source = fixtures_root / 'buggy'
external_source = fixtures_root / 'clean'
dataset_root = Path('/tmp/rtl_phase3_dataset')
manifest_path = dataset_root / 'manifest.json'
build_script_rel = Path('scripts/build_phase3_dataset.py')
build_script = repo_root / build_script_rel
build_script_hint = 'scripts/build_phase3_dataset.py'
build_args = [
    sys.executable,
    str(build_script),
    '--output-dir',
    str(dataset_root),
    '--seed',
    '7',
    '--synthetic-source',
    str(synthetic_source),
    '--external-source',
    str(external_source),
]

display_args = [
    build_script_hint,
    '--output-dir',
    dataset_root.as_posix(),
    '--seed',
    '7',
    '--synthetic-source',
    repo_relative(synthetic_source),
    '--external-source',
    repo_relative(external_source),
]
print('Command:')
print(' '.join(display_args))
result = run_command(build_args)
print(result.stdout.strip())
manifest = read_manifest(manifest_path)
print({'manifest': manifest_path.as_posix(), 'entries': len(manifest.entries), 'seed': manifest.seed})

Command:
scripts/build_phase3_dataset.py --output-dir /tmp/rtl_phase3_dataset --seed 7 --synthetic-source tests/fixtures/buggy --external-source tests/fixtures/clean


dataset manifest written to /tmp/rtl_phase3_dataset/manifest.json
entries: 20 seed: 7
{'manifest': '/tmp/rtl_phase3_dataset/manifest.json', 'entries': 20, 'seed': 7}


## Inspect the manifest and split counts

The manifest is the handoff boundary between downloaded/raw corpora and later training code. Analyzer logic does not reach back into the source trees directly; it consumes the materialized dataset plus manifest metadata.

In [10]:
from collections import Counter

split_counts = Counter(entry.split for entry in manifest.entries)
source_type_counts = Counter(entry.source_type for entry in manifest.entries)
label_counts = Counter(label for entry in manifest.entries for label in entry.labels)
summary = {
    'split_counts': dict(sorted(split_counts.items())),
    'source_type_counts': dict(sorted(source_type_counts.items())),
    'label_counts': dict(sorted(label_counts.items())),
}
print(json.dumps(summary, indent=2, sort_keys=True))

{
  "label_counts": {
    "buggy": 12,
    "clean": 8
  },
  "source_type_counts": {
    "external": 8,
    "synthetic": 12
  },
  "split_counts": {
    "test": 3,
    "train": 14,
    "val": 3
  }
}


In [11]:
manifest_preview = [
    {
        'sample_id': entry.sample_id,
        'split': entry.split,
        'source_type': entry.source_type,
        'path': entry.path,
        'metadata_source_path': entry.metadata.get('source_path'),
        'sha256': entry.sha256[:12],
    }
    for entry in manifest.entries[:5]
]
print(json.dumps(manifest_preview, indent=2))

[
  {
    "sample_id": "synthetic:buggy_cdc.v",
    "split": "train",
    "source_type": "synthetic",
    "path": "train/synthetic/buggy_cdc.v",
    "metadata_source_path": "buggy_cdc.v",
    "sha256": "8f6c828295b9"
  },
  {
    "sample_id": "synthetic:buggy_combo_loop.v",
    "split": "train",
    "source_type": "synthetic",
    "path": "train/synthetic/buggy_combo_loop.v",
    "metadata_source_path": "buggy_combo_loop.v",
    "sha256": "6555e6019e35"
  },
  {
    "sample_id": "synthetic:buggy_counter.v",
    "split": "test",
    "source_type": "synthetic",
    "path": "test/synthetic/buggy_counter.v",
    "metadata_source_path": "buggy_counter.v",
    "sha256": "23cd132abb46"
  },
  {
    "sample_id": "synthetic:buggy_empty_always.v",
    "split": "train",
    "source_type": "synthetic",
    "path": "train/synthetic/buggy_empty_always.v",
    "metadata_source_path": "buggy_empty_always.v",
    "sha256": "7fbae8649260"
  },
  {
    "sample_id": "synthetic:buggy_fsm.sv",
    "split": 

## Check provenance and portability

Downloaded or mirrored corpora feed the manifest build through source directories. The build script records source-relative provenance in `entry.metadata['source_path']`, while `entry.path` points at the copied dataset location under `train/`, `val/`, or `test/`. That keeps provenance explicit without copying corpus-specific logic into the analyzer itself.

In [12]:
rerun_commands = [
    'python -m pytest tests/test_phase3_dataset.py::test_build_script_writes_manifest_and_split_directories -v',
    'python -m pytest tests/test_phase3_docs.py::test_notebook_02_exists_and_mentions_manifest_pipeline -v',
]
print(json.dumps(rerun_commands, indent=2))


[
  "python -m pytest tests/test_phase3_dataset.py::test_build_script_writes_manifest_and_split_directories -v",
  "python -m pytest tests/test_phase3_docs.py::test_notebook_02_exists_and_mentions_manifest_pipeline -v"
]


In [13]:
provenance_checks = {
    'all_manifest_paths_are_relative': all(not Path(entry.path).is_absolute() for entry in manifest.entries),
    'all_source_paths_are_relative': all(not Path(str(entry.metadata['source_path'])).is_absolute() for entry in manifest.entries),
    'all_materialized_files_exist': all((dataset_root / entry.path).exists() for entry in manifest.entries),
    'source_groups_keep_corpus_origin': sorted({entry.source_group for entry in manifest.entries})[:4],
}
print(json.dumps(provenance_checks, indent=2, sort_keys=True))

{
  "all_manifest_paths_are_relative": true,
  "all_materialized_files_exist": true,
  "all_source_paths_are_relative": true,
  "source_groups_keep_corpus_origin": [
    "external:clean_branch_split_feedback.v",
    "external:clean_cdc_synced.v",
    "external:clean_counter.v",
    "external:clean_fsm.v"
  ]
}


In [14]:
materialized_roots = sorted({Path(entry.path).parts[1] for entry in manifest.entries})
print({
    'synthetic_source': repo_relative(synthetic_source),
    'external_source': repo_relative(external_source),
    'materialized_dataset_root': dataset_root.as_posix(),
    'materialized_source_types': materialized_roots,
})

{'synthetic_source': 'tests/fixtures/buggy', 'external_source': 'tests/fixtures/clean', 'materialized_dataset_root': '/tmp/rtl_phase3_dataset', 'materialized_source_types': ['external', 'synthetic']}


## Rerun paths

Rebuild the dataset manually:

```bash
python scripts/build_phase3_dataset.py --output-dir /tmp/rtl_phase3_dataset --seed 7 --synthetic-source tests/fixtures/buggy --external-source tests/fixtures/clean
```

Re-execute this notebook and the focused tests:

```bash
python -m jupyter nbconvert --to notebook --execute --inplace notebooks/rtl_analyzer_phase3/02_dataset_build_and_inspection.ipynb
python -m pytest tests/test_phase3_dataset.py::test_build_script_writes_manifest_and_split_directories -v
python -m pytest tests/test_phase3_docs.py::test_notebook_02_exists_and_mentions_manifest_pipeline -v
```

The key production files for this path are `scripts/build_phase3_dataset.py` and `rtl_analyzer/ml/dataset_manifest.py`.